# Classification Metrics

**Interview Focus**: Precision/recall tradeoffs, ROC vs PR curves, imbalanced data.

**Key Concepts**:
- Accuracy, precision, recall, F1 (binary + multi-class)
- Confusion matrix, ROC curve, AUC-ROC, PR curve, AUC-PR
- When to use which metric

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

## 1. Confusion Matrix Fundamentals

```
                  Predicted
                Pos     Neg
Actual  Pos  [  TP  |  FN  ]
        Neg  [  FP  |  TN  ]
```

From this, all metrics follow.

In [ ]:
def confusion_matrix(y_true, y_pred, n_classes=None):
    """Compute confusion matrix from scratch."""
    if n_classes is None:
        n_classes = max(max(y_true), max(y_pred)) + 1
    
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for true, pred in zip(y_true, y_pred):
        cm[true][pred] += 1
    return cm


def plot_confusion_matrix(cm, class_names=None):
    """Plot confusion matrix as heatmap."""
    n = cm.shape[0]
    if class_names is None:
        class_names = [str(i) for i in range(n)]
    
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im)
    
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title('Confusion Matrix')
    
    # Annotate cells
    for i in range(n):
        for j in range(n):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=14)
    
    plt.tight_layout()
    return fig


# Demo: binary classification
np.random.seed(42)
y_true = np.array([1]*100 + [0]*900)  # imbalanced: 10% positive
# Simulate a model that's pretty good but has some errors
y_pred = y_true.copy()
# Add noise
flip_idx = np.random.choice(len(y_true), size=80, replace=False)
y_pred[flip_idx] = 1 - y_pred[flip_idx]

cm = confusion_matrix(y_true, y_pred)
print(f"Confusion Matrix:\n{cm}")
print(f"\nTP={cm[1,1]}, FN={cm[1,0]}, FP={cm[0,1]}, TN={cm[0,0]}")

fig = plot_confusion_matrix(cm, class_names=['Negative', 'Positive'])
plt.show()

## 2. Core Metrics from Scratch

In [ ]:
def accuracy(y_true, y_pred):
    """(TP + TN) / total"""
    return np.mean(np.array(y_true) == np.array(y_pred))


def precision_score(y_true, y_pred, pos_label=1):
    """TP / (TP + FP) — Of all predicted positive, how many are actually positive?"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    tp = np.sum((y_pred == pos_label) & (y_true == pos_label))
    fp = np.sum((y_pred == pos_label) & (y_true != pos_label))
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0


def recall_score(y_true, y_pred, pos_label=1):
    """TP / (TP + FN) — Of all actual positives, how many did we catch?"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    tp = np.sum((y_pred == pos_label) & (y_true == pos_label))
    fn = np.sum((y_pred != pos_label) & (y_true == pos_label))
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0


def f1_score(y_true, y_pred, pos_label=1):
    """Harmonic mean of precision and recall: 2 * P * R / (P + R)"""
    p = precision_score(y_true, y_pred, pos_label)
    r = recall_score(y_true, y_pred, pos_label)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


# Compute metrics
print(f"Accuracy:  {accuracy(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_true, y_pred):.4f}")
print(f"\nNote: Accuracy is {accuracy(y_true, y_pred):.1%} but F1 is only {f1_score(y_true, y_pred):.1%}")
print("On imbalanced data, accuracy is misleading.")

## 3. Multi-Class F1: Macro, Micro, Weighted

In [ ]:
def precision_recall_f1_per_class(y_true, y_pred):
    """Compute precision, recall, F1 for each class."""
    classes = sorted(set(y_true) | set(y_pred))
    results = {}
    
    for cls in classes:
        p = precision_score(y_true, y_pred, pos_label=cls)
        r = recall_score(y_true, y_pred, pos_label=cls)
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        support = sum(1 for y in y_true if y == cls)
        results[cls] = {'precision': p, 'recall': r, 'f1': f1, 'support': support}
    
    return results


def f1_macro(y_true, y_pred):
    """Average F1 across classes (treats all classes equally)."""
    per_class = precision_recall_f1_per_class(y_true, y_pred)
    return np.mean([v['f1'] for v in per_class.values()])


def f1_micro(y_true, y_pred):
    """Global TP, FP, FN then compute F1 (equivalent to accuracy for multi-class)."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    classes = sorted(set(y_true) | set(y_pred))
    
    tp_total = sum(np.sum((y_pred == c) & (y_true == c)) for c in classes)
    fp_total = sum(np.sum((y_pred == c) & (y_true != c)) for c in classes)
    fn_total = sum(np.sum((y_pred != c) & (y_true == c)) for c in classes)
    
    precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) > 0 else 0
    recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) > 0 else 0
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0


def f1_weighted(y_true, y_pred):
    """Weighted average F1 by class support (accounts for class imbalance)."""
    per_class = precision_recall_f1_per_class(y_true, y_pred)
    total_support = sum(v['support'] for v in per_class.values())
    return sum(v['f1'] * v['support'] / total_support for v in per_class.values())


# Multi-class demo
np.random.seed(42)
y_true_mc = np.array([0]*300 + [1]*100 + [2]*50)  # imbalanced 3-class
y_pred_mc = y_true_mc.copy()
flip = np.random.choice(len(y_true_mc), 60, replace=False)
y_pred_mc[flip] = np.random.randint(0, 3, 60)

print("Per-class metrics:")
per_class = precision_recall_f1_per_class(y_true_mc, y_pred_mc)
print(f"{'Class':>8} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
for cls, metrics in per_class.items():
    print(f"{cls:>8} {metrics['precision']:>10.4f} {metrics['recall']:>10.4f} "
          f"{metrics['f1']:>10.4f} {metrics['support']:>10}")

print(f"\nF1 Macro:    {f1_macro(y_true_mc, y_pred_mc):.4f} (treats all classes equally)")
print(f"F1 Micro:    {f1_micro(y_true_mc, y_pred_mc):.4f} (= accuracy for multi-class)")
print(f"F1 Weighted: {f1_weighted(y_true_mc, y_pred_mc):.4f} (weighted by class frequency)")

## 4. ROC Curve and AUC-ROC

In [ ]:
def roc_curve(y_true, y_scores):
    """Compute ROC curve: TPR vs FPR at various thresholds."""
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)
    
    # Sort by score descending
    sorted_indices = np.argsort(-y_scores)
    y_true_sorted = y_true[sorted_indices]
    y_scores_sorted = y_scores[sorted_indices]
    
    # Get unique thresholds
    thresholds = np.unique(y_scores_sorted)
    thresholds = np.sort(thresholds)[::-1]  # descending
    
    tprs = [0.0]  # start at (0, 0)
    fprs = [0.0]
    
    total_pos = np.sum(y_true == 1)
    total_neg = np.sum(y_true == 0)
    
    for thresh in thresholds:
        y_pred = (y_scores >= thresh).astype(int)
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        
        tpr = tp / total_pos if total_pos > 0 else 0
        fpr = fp / total_neg if total_neg > 0 else 0
        
        tprs.append(tpr)
        fprs.append(fpr)
    
    return np.array(fprs), np.array(tprs), thresholds


def auc_trapezoid(x, y):
    """Compute AUC using the trapezoidal rule."""
    # Sort by x
    sorted_indices = np.argsort(x)
    x = x[sorted_indices]
    y = y[sorted_indices]
    
    # Trapezoidal rule
    area = 0.0
    for i in range(1, len(x)):
        area += (x[i] - x[i-1]) * (y[i] + y[i-1]) / 2
    return area


# Generate scores for an imbalanced classification
np.random.seed(42)
n_pos, n_neg = 100, 900
y_true_binary = np.array([1]*n_pos + [0]*n_neg)

# Good model: positive examples get higher scores
scores_good = np.concatenate([
    np.random.beta(5, 2, n_pos),   # positive: higher scores
    np.random.beta(2, 5, n_neg),   # negative: lower scores
])

# Mediocre model
scores_mediocre = np.concatenate([
    np.random.beta(3, 2, n_pos),
    np.random.beta(2, 3, n_neg),
])

# Random model
scores_random = np.random.rand(n_pos + n_neg)

# Compute ROC curves
fpr_good, tpr_good, _ = roc_curve(y_true_binary, scores_good)
fpr_med, tpr_med, _ = roc_curve(y_true_binary, scores_mediocre)
fpr_rand, tpr_rand, _ = roc_curve(y_true_binary, scores_random)

auc_good = auc_trapezoid(fpr_good, tpr_good)
auc_med = auc_trapezoid(fpr_med, tpr_med)
auc_rand = auc_trapezoid(fpr_rand, tpr_rand)

print(f"AUC-ROC: Good={auc_good:.3f}, Mediocre={auc_med:.3f}, Random={auc_rand:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC Curve
ax = axes[0]
ax.plot(fpr_good, tpr_good, 'b-', linewidth=2, label=f'Good (AUC={auc_good:.3f})')
ax.plot(fpr_med, tpr_med, 'orange', linewidth=2, label=f'Mediocre (AUC={auc_med:.3f})')
ax.plot(fpr_rand, tpr_rand, 'r--', linewidth=1, label=f'Random (AUC={auc_rand:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

# Score distributions
ax = axes[1]
ax.hist(scores_good[:n_pos], bins=30, alpha=0.5, label='Positive', color='blue', density=True)
ax.hist(scores_good[n_pos:], bins=30, alpha=0.5, label='Negative', color='red', density=True)
ax.set_xlabel('Score')
ax.set_ylabel('Density')
ax.set_title('Score Distribution (Good Model)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. PR Curve and AUC-PR

In [ ]:
def pr_curve(y_true, y_scores):
    """Compute Precision-Recall curve."""
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)
    
    sorted_indices = np.argsort(-y_scores)
    y_true_sorted = y_true[sorted_indices]
    
    precisions = []
    recalls = []
    
    tp = 0
    total_pos = np.sum(y_true == 1)
    
    for i in range(len(y_true_sorted)):
        if y_true_sorted[i] == 1:
            tp += 1
        
        precision = tp / (i + 1)
        recall = tp / total_pos if total_pos > 0 else 0
        
        precisions.append(precision)
        recalls.append(recall)
    
    return np.array(recalls), np.array(precisions)


# Compute PR curves
recall_good, prec_good = pr_curve(y_true_binary, scores_good)
recall_med, prec_med = pr_curve(y_true_binary, scores_mediocre)
recall_rand, prec_rand = pr_curve(y_true_binary, scores_random)

auc_pr_good = auc_trapezoid(recall_good, prec_good)
auc_pr_med = auc_trapezoid(recall_med, prec_med)
auc_pr_rand = auc_trapezoid(recall_rand, prec_rand)

# Baseline for PR curve = prevalence (positive rate)
prevalence = n_pos / (n_pos + n_neg)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall_good, prec_good, 'b-', linewidth=2, label=f'Good (AUC-PR={auc_pr_good:.3f})')
ax.plot(recall_med, prec_med, 'orange', linewidth=2, label=f'Mediocre (AUC-PR={auc_pr_med:.3f})')
ax.plot(recall_rand, prec_rand, 'r--', linewidth=1, label=f'Random (AUC-PR={auc_pr_rand:.3f})')
ax.axhline(y=prevalence, color='k', linestyle='--', alpha=0.3, label=f'Random baseline ({prevalence:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.show()

print(f"AUC-PR: Good={auc_pr_good:.3f}, Mediocre={auc_pr_med:.3f}, Random={auc_pr_rand:.3f}")
print(f"\nPR curve is more informative for imbalanced data (prevalence={prevalence:.1%}).")

## 6. Multi-Class ROC (One-vs-Rest)

In [ ]:
# Multi-class ROC: treat each class as binary (one-vs-rest)
np.random.seed(42)
n_classes = 3
n_samples = 500

y_true_mc = np.random.choice(n_classes, n_samples, p=[0.5, 0.3, 0.2])

# Simulate probability scores
scores_mc = np.random.dirichlet([2, 2, 2], n_samples)
# Boost the correct class score
for i in range(n_samples):
    scores_mc[i, y_true_mc[i]] += np.random.uniform(0.5, 1.5)
# Normalize to probabilities
scores_mc = scores_mc / scores_mc.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 5))
class_names = ['Class 0', 'Class 1', 'Class 2']
colors = ['#2196F3', '#FF9800', '#4CAF50']

for cls in range(n_classes):
    # One-vs-Rest: binary problem for this class
    y_binary = (y_true_mc == cls).astype(int)
    fprs, tprs, _ = roc_curve(y_binary, scores_mc[:, cls])
    auc_val = auc_trapezoid(fprs, tprs)
    ax.plot(fprs, tprs, color=colors[cls], linewidth=2, 
            label=f'{class_names[cls]} (AUC={auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Multi-Class ROC (One-vs-Rest)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Imbalanced Data Deep Dive

In [ ]:
# Demonstrate how accuracy is misleading on imbalanced data

# Scenario: 1% positive rate (e.g., fraud detection)
np.random.seed(42)
n_total = 10000
n_pos = 100
y_true_imb = np.array([1]*n_pos + [0]*(n_total - n_pos))

# Model A: always predicts negative (useless but high accuracy)
y_pred_a = np.zeros(n_total, dtype=int)

# Model B: catches 70% of positives with some false positives
y_pred_b = np.zeros(n_total, dtype=int)
# True positives: 70 out of 100 positives
y_pred_b[:70] = 1
# False positives: 200 false alarms
false_pos_idx = np.random.choice(range(n_pos, n_total), 200, replace=False)
y_pred_b[false_pos_idx] = 1

print(f"{'Metric':<15} {'Model A (always neg)':>22} {'Model B (useful)':>22}")
print("-" * 62)
print(f"{'Accuracy':<15} {accuracy(y_true_imb, y_pred_a):>22.4f} {accuracy(y_true_imb, y_pred_b):>22.4f}")
print(f"{'Precision':<15} {precision_score(y_true_imb, y_pred_a):>22.4f} {precision_score(y_true_imb, y_pred_b):>22.4f}")
print(f"{'Recall':<15} {recall_score(y_true_imb, y_pred_a):>22.4f} {recall_score(y_true_imb, y_pred_b):>22.4f}")
print(f"{'F1':<15} {f1_score(y_true_imb, y_pred_a):>22.4f} {f1_score(y_true_imb, y_pred_b):>22.4f}")

print(f"\nModel A has {accuracy(y_true_imb, y_pred_a):.1%} accuracy but is useless (catches 0 fraud).")
print(f"Model B has lower accuracy but catches {recall_score(y_true_imb, y_pred_b):.0%} of fraud.")
print(f"F1 score correctly identifies Model B as better.")

## Interview Questions & Answers

---

**Q: Precision vs Recall — when to prioritize which?**

- **Prioritize Precision**: When false positives are costly. Example: spam filter (don't want to mark real email as spam), content recommendation (bad recommendations hurt user trust).
- **Prioritize Recall**: When false negatives are costly. Example: cancer screening (don't want to miss a cancer case), fraud detection (missing fraud is worse than a false alarm).
- F1 balances both. Use F-beta with beta > 1 for recall emphasis, beta < 1 for precision emphasis.

---

**Q: ROC vs PR curve — when is PR better?**

- **PR curve is better for imbalanced data**. ROC can be overly optimistic when negatives dominate (FPR denominator is huge, making FPR look small even with many false positives).
- ROC AUC: good baseline metric, threshold-independent. Random classifier = 0.5.
- PR AUC: more sensitive to improvements on the positive class. Random classifier = prevalence (positive rate).
- Rule of thumb: if positive rate < 5%, prefer PR curve.

---

**Q: F1 vs accuracy on imbalanced data?**

Accuracy counts TN, which inflates the score when negatives dominate. A model predicting all negatives gets 99% accuracy on a 1% positive rate. F1 ignores TN and focuses on the positive class:

```
F1 = 2 * TP / (2*TP + FP + FN)
```

Use F1 macro for multi-class imbalanced problems (gives equal weight to each class regardless of frequency).

---

**Q: Macro vs Micro vs Weighted F1?**

- **Macro**: Average F1 across classes. Treats rare classes equally. Use when all classes matter equally.
- **Micro**: Global TP/FP/FN. Equivalent to accuracy for multi-class. Use when overall correctness matters.
- **Weighted**: Average F1 weighted by class frequency. Compromise between macro and micro.

## Quick Reference

```
Accuracy  = (TP + TN) / (TP + TN + FP + FN)
Precision = TP / (TP + FP)      — "Of predicted positive, how many correct?"
Recall    = TP / (TP + FN)      — "Of actual positive, how many caught?"
F1        = 2PR / (P + R)       — Harmonic mean of P and R

ROC curve:  TPR vs FPR          — Threshold-independent, always baseline 0.5
PR curve:   Precision vs Recall — Better for imbalanced data
```